# Task 3 trên Kaggle GPU T4 ×2 — không cần API key

Chọn Settings → Accelerator → GPU T4 ×2, bật Internet. Add Input bộ mã nguồn task3_kaggle_source (ZIP hoặc đã giải nén), sample_dataset đủ 32 mẫu, public test và requested levels như notebook API.

Model: Qwen/Qwen3-8B, NF4 4-bit, FP16, tắt thinking. Mỗi GPU chạy một bản model trên một nửa dữ liệu. Đây là inference với model gốc + RAG từ 32 mẫu, không phải fine-tune. Không có quota API nhưng vẫn chịu giới hạn thời gian GPU Kaggle.

Chạy smoke trước để kiểm tra VRAM, chẩn đoán và tiếng Việt. Chưa đo điểm leaderboard trên T4. Bản v5 lưu checkpoint từng bài và bật fallback sau khi sửa feedback thất bại; mọi fallback được đếm riêng. Nếu hết VRAM, giảm max_input_tokens và dùng zero_shot để bỏ ví dụ RAG; không cắt âm thầm code sinh viên.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, json, os, shutil

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
REPO = WORK / 'LLM-APA-task3'

# Kaggle có thể tự giải nén ZIP khi upload.
sources = sorted({p.parent.parent for p in INPUT.rglob('scripts/predict.py')
                  if (p.parent.parent / 'src/llm_grading').is_dir()
                  and (p.parent.parent / 'tasks/task3_feedback/prompts/v004.txt').is_file()})
archives = list(INPUT.rglob('task3_kaggle_source.zip'))
if len(sources) == 1:
    shutil.copytree(sources[0], REPO, dirs_exist_ok=True)
    print('Đã copy mã nguồn từ:', sources[0])
elif not sources and len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(REPO)
    print('Đã giải nén:', archives[0])
else:
    raise ValueError(f'Cần một bộ mã nguồn duy nhất. Thư mục: {sources}; ZIP: {archives}')

assert (REPO/'tasks/task3_feedback/prompts/v005.txt').is_file(), 'Cập nhật task3_kaggle_source.zip mới (prompt v005)'
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'pyyaml', 'scikit-learn', 'tqdm', 'transformers>=4.56,<5', 'accelerate>=1.0,<2', 'bitsandbytes>=0.46,<1'], check=True)
sys.path[:0] = [str(REPO), str(REPO / 'src')]

def run(script, *args):
    subprocess.run([sys.executable, str(REPO/'scripts'/script), *map(str,args)], check=True)


Nếu cell cài thư viện yêu cầu restart kernel, hãy restart rồi chạy lại các cell trước khi tiếp tục. Giữ PyTorch CUDA có sẵn của Kaggle, không chạy uv sync --all-groups ở notebook này.


In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Chưa bật GPU'
assert torch.cuda.device_count() == 2, 'Chọn GPU T4 x2 trong Settings'
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), 'GiB')


In [ ]:
# Nếu tìm thấy nhiều dataset, điền đường dẫn chính xác ở đây.
TRAIN_JSON = None  # '/kaggle/input/.../sample_dataset/task3_feedback.json'
PUBLIC_JSON = None # '/kaggle/input/.../public_test_release/task3_feedback.json'
LEVELS_JSON = None # '/kaggle/input/.../task3_requested_levels....json'

public_archives = list(INPUT.rglob('public_test.zip'))
if PUBLIC_JSON is None and public_archives:
    assert len(public_archives) == 1
    with zipfile.ZipFile(public_archives[0]) as z:
        z.extractall(WORK/'public-data')
files = list(INPUT.rglob('task3_feedback.json')) + list((WORK/'public-data').rglob('task3_feedback.json'))
def choose(training):
    found=[]
    for p in files:
        rows=json.loads(p.read_text(encoding='utf-8-sig'))['samples']
        if bool(rows and 'output' in rows[0]) == training:
            found.append(p)
    assert len(found)==1, f'Điền đường dẫn thủ công, ứng viên: {found}'
    return found[0]
TRAIN_JSON = Path(TRAIN_JSON) if TRAIN_JSON else choose(True)
PUBLIC_JSON = Path(PUBLIC_JSON) if PUBLIC_JSON else choose(False)
if LEVELS_JSON is None:
    matches=list(INPUT.rglob('task3_requested_levels*.json'))
    assert len(matches)==1, 'Cần file requested levels duy nhất'
    LEVELS_JSON=matches[0]
rows=json.loads(Path(LEVELS_JSON).read_text(encoding='utf-8-sig'))
levels={r['sample_id']:r['requested_level'] for r in rows}
assert len(levels)==len(rows) and all(type(v) is int and v in (1,2,3,4) for v in levels.values())
public=json.loads(PUBLIC_JSON.read_text(encoding='utf-8-sig'))['samples']
assert set(levels)=={r['sample_id'] for r in public}
for r in public:
    assert int(r['input']['target_feedback_level'].split()[1]) == levels[r['sample_id']], r['sample_id']
from llm_grading.data.loader import load_samples
train=load_samples(TRAIN_JSON, task='task3')
assert len(train)==32, f'Train phải có 32 mẫu, thấy {len(train)}'
print('Train:', len(train), 'Public:', len(public), 'Level khớp toàn bộ')


In [ ]:
import yaml
from huggingface_hub import HfApi, snapshot_download

MODEL = 'Qwen/Qwen3-8B'
OUT = WORK/'task3-qwen3-8b-t4-v5'
OUT.mkdir(exist_ok=True)
# Ghim revision thực tế và tải model một lần trước khi mở hai worker.
REVISION = HfApi().model_info(MODEL).sha
snapshot_download(MODEL, revision=REVISION,
    allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', '*.jinja'])

cfg = yaml.safe_load((REPO/'configs/task3/rag.yaml').read_text(encoding='utf-8'))
cfg['experiment'].update(id='task3-qwen3-8b-t4-v5', name='task3-qwen3-8b-t4-v5')
cfg['prompt']['version'] = 'v005'
cfg['model'] = {'provider':'local', 'name_or_path':MODEL, 'revision':REVISION,
    'dtype':'float16', 'attn_implementation':'sdpa', 'adapter_path':None}
cfg['quantization'] = {'load_in_4bit':True, 'quant_type':'nf4', 'double_quant':True}
cfg['data'].update(train_path=str(TRAIN_JSON), input_path=str(TRAIN_JSON))
cfg['retrieval'].update(index='bm25', key='code', k=1, compact_examples=True,
    exclude_duplicates=True, exclude_same_exam=False)
cfg['task3'].update(two_stage=True, max_compliance_retries=2,
    fallback_on_failure=True, use_grading_policy=False)
cfg['task2'] = {'use_grading_policy':False}
cfg['generation'].update(max_input_tokens=8192, max_new_tokens=1024,
    do_sample=False, temperature=0, num_beams=1)
cfg['saving']['output_dir'] = str(OUT/'smoke')
CONFIG = OUT/'smoke-config.json'
CONFIG.write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')
print('Model:', MODEL, 'revision:', REVISION)


In [ ]:
# Smoke trên GPU 0. Không nạp model vào kernel notebook.
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false',
           PYTHONUNBUFFERED='1')
subprocess.run([sys.executable, '-u', str(REPO/'scripts/predict.py'),
    '--config', str(CONFIG), '--input', str(PUBLIC_JSON), '--max-samples','3',
    '--output',str(OUT/'smoke_predictions.json')], env=env, check=True)
print((OUT/'smoke_predictions.json').read_text(encoding='utf-8'))


## Chạy cả hai GPU

Hai worker chạy shard 1/2 và 2/2. Mỗi shard có config, log và cache riêng. Cell hiển thị các dòng tiến độ từ cả hai GPU. Giữ lại toàn bộ thư mục task3-qwen3-8b-t4 nếu cần chạy lại: cache chỉ tái sử dụng request đã hoàn tất trong cùng cấu hình/model revision.


In [ ]:
import copy
import selectors

workers = []
selector = selectors.DefaultSelector()
try:
    for gpu in range(2):
        shard_dir = OUT/f'shard-{gpu+1}'
        shard_dir.mkdir(exist_ok=True)
        shard_cfg = copy.deepcopy(cfg)
        shard_cfg['saving']['output_dir'] = str(shard_dir)
        shard_cfg['model']['cache_path'] = str(shard_dir/'generation_cache.jsonl')
        config_path = shard_dir/'config.json'
        config_path.write_text(json.dumps(shard_cfg,ensure_ascii=False,indent=2),encoding='utf-8')
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1')
        cmd = [sys.executable,'-u',str(REPO/'scripts/predict.py'),
            '--config',str(config_path),'--input',str(PUBLIC_JSON),
            '--shard',f'{gpu+1}/2','--resume','--output',str(shard_dir/'predictions.json')]
        p = subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        workers.append(p)
        selector.register(p.stdout, selectors.EVENT_READ, gpu)
    while selector.get_map():
        for key, _ in selector.select(timeout=1):
            line = key.fileobj.readline()
            if line:
                print(f'[GPU {key.data}] {line}',end='',flush=True)
            else:
                selector.unregister(key.fileobj)
                key.fileobj.close()
    codes = [p.wait() for p in workers]
    assert codes == [0,0], f'Worker lỗi: {codes}; xem log phía trên, chưa đóng gói kết quả.'
finally:
    selector.close()
    for p in workers:
        if p.poll() is None:
            p.terminate()
    for p in workers:
        p.wait()


In [ ]:
run('merge_predictions.py','--task','task3',
    '--shards',OUT/'shard-1/predictions.json',OUT/'shard-2/predictions.json',
    '--input',PUBLIC_JSON,'--output',OUT/'predictions.json')
pred=json.loads((OUT/'predictions.json').read_text(encoding='utf-8'))
assert {r['sample_id'] for r in pred} == set(levels)
assert all(r['output']['feedback_level']==levels[r['sample_id']] for r in pred)
run('validate_predictions.py','--task','task3','--input',OUT/'predictions.json')
run('package_predictions.py','--task','task3','--input',OUT/'predictions.json',
    '--output',WORK/'submission_task3_qwen3_8b.zip')
print('Tải /kaggle/working/submission_task3_qwen3_8b.zip. Giữ logs/cache riêng tư.')


In [ ]:
# Báo cáo chất lượng vận hành; fallback không đồng nghĩa chẩn đoán tốt.
from collections import Counter
fallbacks = [r for r in pred if r['output'].get('fallback')]
print('Tổng bài:', len(pred), '| Fallback:', len(fallbacks))
print('Fallback theo level:', dict(Counter(r['output']['feedback_level'] for r in fallbacks)))
(OUT/'fallback_report.json').write_text(json.dumps({
    'count':len(fallbacks), 'sample_ids':[r['sample_id'] for r in fallbacks]
},ensure_ascii=False,indent=2),encoding='utf-8')
print('Xem nguyên nhân trong shard-1/responses.jsonl và shard-2/responses.jsonl')
